
# SHA-256 Reverse Tension Probe

This notebook implements **two probes** for a single-block SHA-256 reverse game:

1. **Geometry probe** — a true side-data **hot/cold** score.  
   It never compares your guess to the hidden schedule word directly.  
   It scores a guess only by **internal carry geometry** and other admissible side observables.

2. **Verifier probe** — an optional hidden answer check.  
   This is **not** geometry-only. It exists only to measure whether the geometry probe is useful.

The forward pass computes the full block internally, but the exported geometry package exposes only:

- digest
- per-round staged carry-out bits
- per-round carry-mask Hamming weights
- per-round `h`-register Hamming weight

No direct `W[t]`, no message words, and no round-state values are exposed in the visible geometry bundle.


In [ ]:

# Optional setup cell.
# No third-party packages are required for this notebook.


In [ ]:

from __future__ import annotations

from dataclasses import dataclass
import hashlib
import os
import random
import struct
from typing import Dict, List, Tuple

MASK32 = 0xFFFFFFFF

H0 = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

def u32(x: int) -> int:
    return x & MASK32

def rotr(x: int, n: int) -> int:
    x &= MASK32
    return ((x >> n) | (x << (32 - n))) & MASK32

def ch(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (~x & z)) & MASK32

def maj(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (x & z) ^ (y & z)) & MASK32

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def hw(x: int) -> int:
    return (x & MASK32).bit_count()

def pad_single_block(msg: bytes) -> bytes:
    if len(msg) > 55:
        raise ValueError("This notebook is restricted to single-block messages of length <= 55 bytes.")
    bit_len = len(msg) * 8
    out = msg + b"\x80"
    while len(out) % 64 != 56:
        out += b"\x00"
    out += struct.pack(">Q", bit_len)
    return out

def words_from_block(block: bytes) -> List[int]:
    return list(struct.unpack(">16I", block))

def expand_schedule(w16: List[int]) -> List[int]:
    W = list(w16)
    for t in range(16, 64):
        W.append(u32(sigma1(W[t - 2]) + W[t - 7] + sigma0(W[t - 15]) + W[t - 16]))
    return W

def hashlib_sha256_hex(msg: bytes) -> str:
    return hashlib.sha256(msg).hexdigest()

def carry_mask_add(a: int, b: int) -> int:
    a &= MASK32
    b &= MASK32
    carry = a & b
    union = carry
    s = a ^ b
    while carry:
        carry = (carry << 1) & MASK32
        newcarry = s & carry
        union |= newcarry
        s ^= carry
        carry = newcarry
    return union

def add32(a: int, b: int) -> Tuple[int, int, int]:
    total = (a & MASK32) + (b & MASK32)
    return total & MASK32, int(total >> 32), carry_mask_add(a, b)

def commitment(secret_salt: bytes, label: str, value: int) -> str:
    payload = secret_salt + label.encode("utf-8") + struct.pack(">I", value & MASK32)
    return hashlib.sha256(payload).hexdigest()


In [ ]:

@dataclass
class RoundTrace:
    t: int
    a: int
    b: int
    c: int
    d: int
    e: int
    f: int
    g: int
    h: int
    Wt: int
    T1: int
    T2: int
    stage_carries: Tuple[int, int, int, int]
    stage_mask_hw: Tuple[int, int, int, int]
    h_hw: int

def compress_single_block_with_trace(msg: bytes) -> Dict[str, object]:
    block = pad_single_block(msg)
    W = expand_schedule(words_from_block(block))

    a, b, c, d, e, f, g, h = H0
    traces: List[RoundTrace] = []

    for t in range(64):
        s1 = Sigma1(e)
        chv = ch(e, f, g)

        sA, c1, m1 = add32(h, s1)
        sB, c2, m2 = add32(sA, chv)
        sC, c3, m3 = add32(sB, K[t])
        T1, c4, m4 = add32(sC, W[t])

        T2 = u32(Sigma0(a) + maj(a, b, c))

        traces.append(
            RoundTrace(
                t=t, a=a, b=b, c=c, d=d, e=e, f=f, g=g, h=h,
                Wt=W[t], T1=T1, T2=T2,
                stage_carries=(c1, c2, c3, c4),
                stage_mask_hw=tuple(hw(m) for m in (m1, m2, m3, m4)),
                h_hw=hw(h),
            )
        )

        new_a = u32(T1 + T2)
        new_e = u32(d + T1)
        a, b, c, d, e, f, g, h = new_a, a, b, c, new_e, e, f, g

    working_final = [a, b, c, d, e, f, g, h]
    digest_words = [u32(H0[i] + working_final[i]) for i in range(8)]
    digest_hex = "".join(f"{x:08x}" for x in digest_words)
    assert digest_hex == hashlib_sha256_hex(msg)

    return {
        "W": W,
        "traces": traces,
        "working_final": working_final,
        "digest_words": digest_words,
        "digest_hex": digest_hex,
    }


In [ ]:

@dataclass
class RoundGeometry:
    t: int
    stage_carries: Tuple[int, int, int, int]
    stage_mask_hw: Tuple[int, int, int, int]
    h_hw: int

@dataclass
class GeometryBundle:
    digest_hex: str
    digest_words: List[int]
    rounds: Dict[int, RoundGeometry]

    def working_final_state(self) -> List[int]:
        return [u32(self.digest_words[i] - H0[i]) for i in range(8)]

@dataclass
class HiddenVerifier:
    secret_salt: bytes
    word_commitments: Dict[int, str]
    true_words: Dict[int, int]

    def verify(self, t: int, guess: int) -> bool:
        return self.word_commitments[t] == commitment(self.secret_salt, f"W[{t}]", guess)

@dataclass
class ProbeBundle:
    geometry: GeometryBundle
    verifier: HiddenVerifier

def build_probe_bundle(msg: bytes) -> ProbeBundle:
    full = compress_single_block_with_trace(msg)
    traces: List[RoundTrace] = full["traces"]
    geom = GeometryBundle(
        digest_hex=full["digest_hex"],
        digest_words=full["digest_words"],
        rounds={
            rt.t: RoundGeometry(
                t=rt.t,
                stage_carries=rt.stage_carries,
                stage_mask_hw=rt.stage_mask_hw,
                h_hw=rt.h_hw,
            )
            for rt in traces
        }
    )
    salt = os.urandom(32)
    verifier = HiddenVerifier(
        secret_salt=salt,
        word_commitments={rt.t: commitment(salt, f"W[{rt.t}]", rt.Wt) for rt in traces},
        true_words={rt.t: rt.Wt for rt in traces},
    )
    return ProbeBundle(geometry=geom, verifier=verifier)



## Reverse step

For a guessed `W[t]`, the next state `x[t+1]` determines most of `x[t]` immediately by the register shifts.

The only real freedom is the guessed schedule word.  
That guess changes the reconstructed `h[t]`, and therefore changes the **internal addition geometry** in the `T1` subcircuit.

That is exactly where the hot/cold probe lives.


In [ ]:

def reverse_step_from_next(next_state: List[int], t: int, W_guess: int) -> Dict[str, object]:
    a1, b1, c1, d1, e1, f1, g1, h1 = next_state

    a_t = b1
    b_t = c1
    c_t = d1
    e_t = f1
    f_t = g1
    g_t = h1

    T2 = u32(Sigma0(a_t) + maj(a_t, b_t, c_t))
    T1 = u32(a1 - T2)
    d_t = u32(e1 - T1)

    s1 = Sigma1(e_t)
    chv = ch(e_t, f_t, g_t)
    const_tail = u32(s1 + chv + K[t])

    h_t = u32(T1 - const_tail - W_guess)

    sA, c1, m1 = add32(h_t, s1)
    sB, c2, m2 = add32(sA, chv)
    sC, c3, m3 = add32(sB, K[t])
    T1_check, c4, m4 = add32(sC, W_guess)

    return {
        "state": [a_t, b_t, c_t, d_t, e_t, f_t, g_t, h_t],
        "T1": T1,
        "T2": T2,
        "T1_check": T1_check,
        "stage_carries": (c1, c2, c3, c4),
        "stage_mask_hw": tuple(hw(m) for m in (m1, m2, m3, m4)),
        "h_hw": hw(h_t),
    }


In [ ]:

def geometry_score(bundle: ProbeBundle, next_state: List[int], t: int, guess: int) -> Dict[str, object]:
    observed = bundle.geometry.rounds[t]
    pred = reverse_step_from_next(next_state, t, guess)

    carry_mismatch = sum(int(a != b) for a, b in zip(pred["stage_carries"], observed.stage_carries))
    mask_hw_error = sum(abs(a - b) for a, b in zip(pred["stage_mask_hw"], observed.stage_mask_hw))
    h_hw_error = abs(pred["h_hw"] - observed.h_hw)

    total = 5 * carry_mismatch + mask_hw_error + h_hw_error

    return {
        "guess": guess & MASK32,
        "score": total,
        "carry_mismatch": carry_mismatch,
        "mask_hw_error": mask_hw_error,
        "h_hw_error": h_hw_error,
        "predicted": pred,
        "observed": observed,
    }

def calibrate_temperature(bundle: ProbeBundle, next_state: List[int], t: int, samples: int = 512, seed: int = 1234) -> Dict[str, object]:
    rng = random.Random(seed)
    scores = [geometry_score(bundle, next_state, t, rng.getrandbits(32))["score"] for _ in range(samples)]
    scores.sort()
    return {
        "samples": samples,
        "min": scores[0],
        "q05": scores[max(0, int(0.05 * (samples - 1)))],
        "q20": scores[max(0, int(0.20 * (samples - 1)))],
        "median": scores[max(0, int(0.50 * (samples - 1)))],
        "q80": scores[max(0, int(0.80 * (samples - 1)))],
        "max": scores[-1],
        "sorted_scores": scores,
    }

def hot_cold_label(score: int, calibration: Dict[str, object]) -> str:
    if score == 0:
        return "ice"
    if score <= calibration["q05"]:
        return "cold"
    if score <= calibration["q20"]:
        return "cool"
    if score <= calibration["median"]:
        return "warm"
    if score <= calibration["q80"]:
        return "hot"
    return "burning"

def probe_guess(bundle: ProbeBundle, next_state: List[int], t: int, guess: int, calibration: Dict[str, object] | None = None) -> Dict[str, object]:
    result = geometry_score(bundle, next_state, t, guess)
    result["temperature"] = hot_cold_label(result["score"], calibration) if calibration else "unscaled"
    result["verified"] = bundle.verifier.verify(t, guess)
    return result



## Search helpers

`sample_search()` tries many random guesses and ranks them by geometry score.

This is not a proof of inversion.  
It is a way to test whether the **side-data field has gradient**.


In [ ]:

def sample_search(bundle: ProbeBundle, next_state: List[int], t: int, samples: int = 2000, seed: int = 1234, calibration: Dict[str, object] | None = None) -> List[Dict[str, object]]:
    rng = random.Random(seed)
    out = []
    for _ in range(samples):
        g = rng.getrandbits(32)
        out.append(probe_guess(bundle, next_state, t, g, calibration=calibration))
    out.sort(key=lambda row: (row["score"], row["guess"]))
    return out

def pretty_probe_row(row: Dict[str, object]) -> Dict[str, object]:
    return {
        "guess_hex": f"{row['guess']:08x}",
        "score": row["score"],
        "temp": row["temperature"],
        "carry_mismatch": row["carry_mismatch"],
        "mask_hw_error": row["mask_hw_error"],
        "h_hw_error": row["h_hw_error"],
        "verified": row["verified"],
    }



## Demo: build a bundle

You can replace `TARGET_MESSAGE` with any message up to 55 bytes.


In [ ]:

TARGET_MESSAGE = b"abc"

bundle = build_probe_bundle(TARGET_MESSAGE)
bundle.geometry.digest_hex


In [ ]:

x64 = bundle.geometry.working_final_state()
t = 63

cal = calibrate_temperature(bundle, x64, t=t, samples=1024, seed=1234)
cal



## Try a few guesses

The first row uses the real word only so the notebook can show the geometry probe and verifier side by side.

That does **not** mean the geometry probe is secretly reading the answer.
The side-by-side comparison is here so you can see whether the hot/cold signal is meaningful.


In [ ]:

true_guess = bundle.verifier.true_words[t]

rows = [
    probe_guess(bundle, x64, t=t, guess=true_guess, calibration=cal),
    probe_guess(bundle, x64, t=t, guess=0, calibration=cal),
    probe_guess(bundle, x64, t=t, guess=0xFFFFFFFF, calibration=cal),
    probe_guess(bundle, x64, t=t, guess=0x12345678, calibration=cal),
]

[pretty_probe_row(r) for r in rows]



## Automated hot/cold sampling

This samples random candidates and shows the best hits under the geometry probe.
If the probe has real tension, the verified word should rank unusually well.


In [ ]:

hits = sample_search(bundle, x64, t=t, samples=5000, seed=2026, calibration=cal)
top10 = [pretty_probe_row(r) for r in hits[:10]]
top10


In [ ]:

true_row = probe_guess(bundle, x64, t=t, guess=true_guess, calibration=cal)
rank = 1 + sum(1 for r in hits if (r["score"], r["guess"]) < (true_row["score"], true_row["guess"]))

pretty_probe_row(true_row), rank



## What the score means

The score is **lower-is-better** and is composed of:

- `carry_mismatch`: staged carry-out bit mismatches
- `mask_hw_error`: mismatch in carry-mask Hamming weights
- `h_hw_error`: mismatch in reconstructed `h[t]` Hamming weight

This is a real side-data probe because it only checks **internal carry geometry** and **register-shape statistics**.

It is **not** a full proof that a low score is correct.
It is a **tension meter**.


In [ ]:

def explain_probe(bundle: ProbeBundle, next_state: List[int], t: int, guess: int, calibration: Dict[str, object]) -> None:
    row = probe_guess(bundle, next_state, t=t, guess=guess, calibration=calibration)
    obs = row["observed"]
    pred = row["predicted"]

    print(f"round t = {t}")
    print(f"guess   = 0x{guess:08x}")
    print(f"score   = {row['score']}  ->  {row['temperature']}")
    print(f"verify  = {row['verified']}")
    print()
    print("observed stage_carries :", obs.stage_carries)
    print("pred stage_carries     :", pred["stage_carries"])
    print("observed stage_mask_hw :", obs.stage_mask_hw)
    print("pred stage_mask_hw     :", pred["stage_mask_hw"])
    print("observed h_hw          :", obs.h_hw)
    print("pred h_hw              :", pred["h_hw"])

explain_probe(bundle, x64, t=t, guess=true_guess, calibration=cal)



## Manual play

Type your own candidate below.

This keeps both probes visible:

- geometry score: hot/cold
- hidden verifier: right/wrong

Once you are satisfied the geometry probe is useful, ignore the verifier and use only the hot/cold side.


In [ ]:

# Enter an 8-hex-digit candidate here.
USER_GUESS_HEX = "00000000"

user_guess = int(USER_GUESS_HEX, 16)
pretty_probe_row(probe_guess(bundle, x64, t=t, guess=user_guess, calibration=cal))



## Notes

- This notebook is restricted to **single-block SHA-256** messages (`len(msg) <= 55`) so the reverse walk stays concrete.
- The geometry probe is intentionally **local**. It measures whether a guess fits the side-data exposed at one round.
- The verifier exists only as a benchmark. It is not part of the geometry-only logic.
- The key empirical question is: **does the low-score region correlate strongly enough with the verified word to be useful?**
